### Create weather_embeddings table:

In [0]:
#CREATE weather_embeddings table:
from pathlib import Path
import sys

REPO_ROOT = Path(
    "/Workspace/Users/tuvu.uwyo@gmail.com/weather_intelligence_databricks"
)

NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
SQL_DIR = REPO_ROOT / "sql"

sys.path.insert(0, str(NOTEBOOKS_DIR))

from lakebase import get_connection

sql_file = SQL_DIR / "02_setup_weather_embeddings.sql"

with open(sql_file, "r") as f:
    sql = f.read()

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
    conn.commit()

print("weather_embeddings table created")

### Config

In [0]:
from pathlib import Path
import sys
import os
from sentence_transformers import SentenceTransformer

REPO_ROOT = Path(
    "/Workspace/Users/tuvu.uwyo@gmail.com/weather_intelligence_databricks"
)

sys.path.insert(0, str(REPO_ROOT))

WEATHER_DOCUMENTS_TABLE = "weather_documents"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
BATCH_SIZE = 32

EMBEDDINGS_TABLE = "weather_embeddings"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"

model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    cache_folder="/tmp/.cache/huggingface",
)

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Chunk overlap: {CHUNK_OVERLAP}")

### Load unembedded weather documents

In [0]:
import lakebase
with lakebase.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"""
            SELECT
                d.id,
                d.location,
                d.source_type,
                d.headline,
                d.narrative_text
            FROM {WEATHER_DOCUMENTS_TABLE} d
            WHERE d.narrative_text IS NOT NULL
              AND TRIM(d.narrative_text) <> ''
              AND NOT EXISTS (
                  SELECT 1
                  FROM {EMBEDDINGS_TABLE} e
                  WHERE e.document_id = d.id
              )
            ORDER BY d.synced_at DESC
            """
        )

        weather_documents = cur.fetchall()

print(f"Loaded {len(weather_documents)} unembedded weather documents.")

In [0]:
weather_documents

### Chunk narrative text

In [0]:
def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    """Split text into overlapping character-based chunks."""

    text = text.strip()
    if not text:
        return []
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    step = chunk_size - chunk_overlap
    chunks = []

    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
    return chunks

chunk_rows = []

for document in weather_documents:
    document_id = document["id"]
    narrative_text = document["narrative_text"]
    chunks = chunk_text(narrative_text)
    for chunk_index, chunk in enumerate(chunks):
        chunk_rows.append(
            {
                "document_id": document_id,
                "chunk_index": chunk_index,
                "chunk_text": chunk,
            }
        )
        
print(
    f"Created {len(chunk_rows)} chunks "
    f"from {len(weather_documents)} documents."
)


In [0]:
chunk_rows

### Compute Embedding

In [0]:
all_embeddings = []

for i in range(0, len(chunk_rows), BATCH_SIZE):
    batch = chunk_rows[i:i + BATCH_SIZE]
    texts = [row["chunk_text"] for row in batch]
    vectors = model.encode(
        texts,
        show_progress_bar=False,
    )
    all_embeddings.extend(vectors.tolist())

    print(
        f"Processed "
        f"{min(i + BATCH_SIZE, len(chunk_rows))}/{len(chunk_rows)} chunks"
    )
print(f"Computed {len(all_embeddings)} embeddings.")

### Insert embeddings into Lakebase

In [0]:
import uuid
from datetime import datetime, timezone
from psycopg2.extras import execute_values

if not chunk_rows:
    print("No new weather documents to embed.")
else:
    created_at = datetime.now(timezone.utc)
    insert_rows = []
    for row, embedding in zip(chunk_rows, all_embeddings):
        vector_value = "[" + ",".join(
            str(float(value)) for value in embedding
        ) + "]"

        insert_rows.append(
            (
                str(uuid.uuid4()),
                row["document_id"],
                row["chunk_index"],
                row["chunk_text"],
                vector_value,
                EMBEDDING_MODEL_NAME,
                created_at,
            )
        )

    insert_sql = f"""
        INSERT INTO {EMBEDDINGS_TABLE} (
            id,
            document_id,
            chunk_index,
            chunk_text,
            embedding,
            model_name,
            created_at
        )
        VALUES %s
        ON CONFLICT (document_id, chunk_index) DO NOTHING
    """

    template = """
        (%s, %s, %s, %s, %s::vector, %s, %s)
    """

    with lakebase.get_connection() as conn:
        with conn.cursor() as cur:
            execute_values(
                cur,
                insert_sql,
                insert_rows,
                template=template,
                page_size=100,
            )
            inserted_count = cur.rowcount
        conn.commit()

    print(f"Inserted {inserted_count} new weather embeddings.")